# 📝 04 — Answer Evaluation (Rubric-Based Scoring)

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook demonstrates the semantic answer evaluation pipeline:

1. **Rubric Loading** — Parse structured marking schemes
2. **Semantic Similarity** — Sentence-BERT embeddings (`all-MiniLM-L6-v2`)
3. **Criterion-Level Scoring** — Match student answers to each criterion
4. **Keyword Detection** — Check for required domain terminology
5. **Score Aggregation** — Compute total marks with confidence

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## 1. Load Rubrics

A rubric contains the question, criteria, keywords, and mark allocation.

In [ ]:
from evaluation import Rubric, AnswerEvaluator

# Load all available rubrics
rubric_dir = Path('../data/samples')
rubric_files = sorted([f for f in rubric_dir.glob('*.json') if 'answer' not in f.name])

rubrics = {}
for rpath in rubric_files:
    with open(rpath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rubric = Rubric.from_dict(data)
    rubrics[rpath.stem] = rubric
    
    # Validate
    warnings = rubric.validate()
    status = '✓' if not warnings else f'⚠ {len(warnings)} warnings'
    print(f'{status}  {rpath.name}')
    print(f'      Question: {rubric.question[:70]}...')
    print(f'      Max marks: {rubric.max_marks}, Criteria: {len(rubric.criteria)}')
    for c in rubric.criteria:
        print(f'        [{c.id}] {c.description[:50]}... ({c.marks} marks)')
    print()

## 2. Initialize the Evaluator

The evaluator uses **Sentence-BERT** (`all-MiniLM-L6-v2`) for semantic similarity.

In [ ]:
print('Loading sentence-transformers model...')
t0 = time.time()

evaluator = AnswerEvaluator(
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=0.55
)

elapsed = time.time() - t0
print(f'Evaluator initialized in {elapsed:.1f}s')
print(f'Model: all-MiniLM-L6-v2 (384-dim embeddings)')
print(f'Similarity threshold: 0.55')

## 3. Evaluate Sample Answers

Let's evaluate different quality answers (good, partial, poor) against rubrics.

In [ ]:
# Load sample answers
answers_path = Path('../data/samples/sample_answers.json')
with open(answers_path, 'r', encoding='utf-8') as f:
    answers_data = json.load(f)

all_results = []

for answer_info in answers_data['answers']:
    answer_id = answer_info['id']
    answer_text = answer_info['answer_text']
    rubric_file = answer_info['rubric_file']
    quality = answer_info['quality']
    expected_range = answer_info['expected_score_range']
    
    # Find the matching rubric
    rubric_stem = Path(rubric_file).stem
    if rubric_stem not in rubrics:
        print(f'⚠ Rubric {rubric_file} not found, skipping {answer_id}')
        continue
    
    rubric = rubrics[rubric_stem]
    
    # Evaluate
    t0 = time.time()
    result = evaluator.evaluate(answer_text, rubric)
    eval_time = time.time() - t0
    
    in_range = expected_range[0] <= result.total_score <= expected_range[1]
    status = '✓' if in_range else '✗'
    
    all_results.append({
        'id': answer_id,
        'quality': quality,
        'score': result.total_score,
        'max_score': result.max_score,
        'expected': expected_range,
        'in_range': in_range,
        'confidence': result.overall_confidence,
        'criteria_results': result.criteria_results,
        'result_obj': result,
        'time': eval_time,
    })
    
    print(f'{status} {answer_id:<35}  {quality:<8}  Score: {result.total_score:.1f}/{result.max_score:.0f}  '
          f'Expected: {expected_range}  Conf: {result.overall_confidence:.3f}  ({eval_time:.2f}s)')

In [ ]:
# Detailed criterion-level breakdown for one answer
if all_results:
    example = all_results[0]
    print(f'\n══════ Detailed Breakdown: {example["id"]} ══════')
    print(f'Quality: {example["quality"]}')
    print(f'Total Score: {example["score"]:.1f} / {example["max_score"]:.0f}')
    print(f'Overall Confidence: {example["confidence"]:.4f}\n')
    
    result_dict = example['result_obj'].to_dict()
    for cr in result_dict.get('criteria_results', []):
        print(f'  [{cr["criterion_id"]}]')
        print(f'    Score: {cr["score"]:.1f} / {cr["max_marks"]:.0f}')
        print(f'    Similarity: {cr.get("similarity", 0):.4f}')
        matched = cr.get('matched_keywords', [])
        print(f'    Keywords matched: {matched if matched else "none"}')
        print()

In [ ]:
# Visualization: scores by quality
if all_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1. Score comparison
    ax = axes[0]
    ids = [r['id'].split('_')[-1] for r in all_results]
    scores = [r['score'] for r in all_results]
    max_scores = [r['max_score'] for r in all_results]
    quality_colors = {'good': '#2ECC71', 'partial': '#F39C12', 'poor': '#E74C3C'}
    colors = [quality_colors.get(r['quality'], '#999') for r in all_results]
    
    bars = ax.bar(range(len(ids)), scores, color=colors, edgecolor='white', alpha=0.85)
    ax.bar(range(len(ids)), [m - s for m, s in zip(max_scores, scores)],
           bottom=scores, color='#E0E0E0', edgecolor='white', alpha=0.4)
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(ids, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Score')
    ax.set_title('AI Scores by Answer')
    
    # Legend
    from matplotlib.patches import Patch
    legend_items = [Patch(color=c, label=q.title()) for q, c in quality_colors.items()]
    ax.legend(handles=legend_items, loc='upper right')
    
    # 2. Score vs Expected Range
    ax = axes[1]
    for i, r in enumerate(all_results):
        expected = r['expected']
        ax.plot([i, i], expected, 'k-', linewidth=3, alpha=0.3)
        ax.plot(i, expected[0], 'v', color='gray', markersize=8)
        ax.plot(i, expected[1], '^', color='gray', markersize=8)
        color = '#2ECC71' if r['in_range'] else '#E74C3C'
        ax.plot(i, r['score'], 'o', color=color, markersize=10, zorder=5)
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(ids, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Score')
    ax.set_title('AI Score vs Expected Range')
    
    # 3. Confidence distribution
    ax = axes[2]
    confs = [r['confidence'] for r in all_results]
    ax.bar(range(len(ids)), confs, color='#9B59B6', edgecolor='white', alpha=0.85)
    ax.axhline(y=0.55, color='red', linestyle='--', alpha=0.5, label='Threshold (0.55)')
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(ids, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Confidence')
    ax.set_title('Evaluation Confidence')
    ax.set_ylim(0, 1)
    ax.legend()
    
    plt.suptitle('Answer Evaluation Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/results/evaluation_results.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Semantic Similarity Deep Dive

Let's explore how the sentence-BERT embeddings capture meaning.

In [ ]:
from evaluation.semantic import semantic_similarity, batch_similarity

# Test semantic similarity with various examples
reference = "Photosynthesis is the process by which plants convert light energy into chemical energy"

test_cases = [
    ("Plants use sunlight to make food through photosynthesis", "Correct paraphrase"),
    ("Photosynthesis converts light energy to chemical energy in plants", "Near-exact match"),
    ("Green plants absorb CO2 and release oxygen", "Partial — different aspect"),
    ("The mitochondria is the powerhouse of the cell", "Wrong topic entirely"),
    ("Plants need water and soil to grow", "Tangentially related"),
    ("", "Empty answer"),
]

print(f'Reference: "{reference}"\n')
print(f'{"Similarity":>10}  {"Category":<25}  Text')
print('─' * 90)

for text, category in test_cases:
    if text:
        sim = semantic_similarity(reference, text)
    else:
        sim = 0.0
    bar = '█' * int(sim * 30)
    print(f'{sim:>10.4f}  {category:<25}  {bar} {text[:50]}')

In [ ]:
# Similarity heatmap: criteria vs answer sentences
if all_results and rubrics:
    # Pick first rubric and first good answer
    first_rubric_key = list(rubrics.keys())[0]
    rubric = rubrics[first_rubric_key]
    
    # Find the matching good answer
    good_answers = [r for r in all_results if r['quality'] == 'good']
    if good_answers:
        answer_text = answers_data['answers'][0]['answer_text']
        sentences = [s.strip() for s in answer_text.replace('\n', '. ').split('.') if s.strip()]
        criteria_texts = [c.description for c in rubric.criteria]
        
        # Compute pairwise similarities
        n_criteria = len(criteria_texts)
        n_sentences = min(len(sentences), 8)  # Limit for visualization
        sim_matrix = np.zeros((n_criteria, n_sentences))
        
        for i, crit in enumerate(criteria_texts):
            for j in range(n_sentences):
                sim_matrix[i, j] = semantic_similarity(crit, sentences[j])
        
        fig, ax = plt.subplots(figsize=(14, 6))
        im = ax.imshow(sim_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
        
        ax.set_yticks(range(n_criteria))
        ax.set_yticklabels([f'C{i+1}: {t[:40]}...' for i, t in enumerate(criteria_texts)], fontsize=9)
        ax.set_xticks(range(n_sentences))
        ax.set_xticklabels([f'S{j+1}' for j in range(n_sentences)])
        ax.set_xlabel('Answer Sentences')
        ax.set_ylabel('Rubric Criteria')
        ax.set_title('Semantic Similarity: Criteria × Answer Sentences', fontsize=14)
        
        # Annotate cells
        for i in range(n_criteria):
            for j in range(n_sentences):
                color = 'white' if sim_matrix[i, j] > 0.5 else 'black'
                ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                        fontsize=9, color=color)
        
        plt.colorbar(im, label='Cosine Similarity')
        plt.tight_layout()
        plt.savefig('../data/results/similarity_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()

## Summary

| Component | Details |
|-----------|---------|  
| Embedding Model | `all-MiniLM-L6-v2` (384-dim, 22M params) |
| Similarity Metric | Cosine similarity |
| Threshold | 0.55 for criterion matching |
| Scoring | Per-criterion weighted by rubric marks |
| Keyword Check | Exact + fuzzy matching for domain terms |

**Key Design Decision:** The examiner remains the final authority — AI scores are advisory.

**Next:** Proceed to `05_feature_engineering.ipynb` for risk prediction features.